# Access an ASAM ODS EXD-API Plugin

## Prepare Python Environment to Access GRPC Service

In [1]:
import os
import pathlib

from ods_exd_api_box.proto import ods_external_data_pb2, ods_external_data_pb2_grpc
import grpc
from google.protobuf.json_format import MessageToJson

## EXD-API

The EXD-API plugin is running as a RPC service at a given URL.
Running `external_data_file.py` will run the plugin at the given URL.

In [2]:
exd_api_plugin_url = "localhost:50051"

## Import Phase

We will open a DuckDB file using the EXD-API and extract the internal structure of the file to import it into the ASAM ODS server.

In [14]:
data_file_path = os.path.abspath("data/test.duckdb")
if not os.path.exists(data_file_path):
    raise Exception("Data file is missing")

In [15]:
import_file_url = pathlib.Path(data_file_path).as_uri()
import_file_parameters = "mode=normalized"
print(import_file_url)

# Will be filled from Structure
access_file_url = None
access_file_parameters = None

file:///workspaces/duckdb/asam_ods_exd_api_duckdb/data/test.duckdb


### Extract Infos from Structure

The structure contains infos about groups and channels to create corresponding measurements, submatrices and measurement_quantities

In [16]:
with grpc.insecure_channel(exd_api_plugin_url) as channel:
    stub = ods_external_data_pb2_grpc.ExternalDataReaderStub(channel)

    # import file into ASAM ODS Server physical storage
    import_identifier = ods_external_data_pb2.Identifier(url=import_file_url, parameters=import_file_parameters)

    import_handle = stub.Open(import_identifier)
    try:
        structure = stub.GetStructure(ods_external_data_pb2.StructureRequest(handle=import_handle))
        print(MessageToJson(structure))

        access_file_url = structure.identifier.url
        access_file_parameters = structure.identifier.parameters

        for group in structure.groups:
            group_id = group.id
            for channel in group.channels:
                channel_id = channel.id
    finally:
        stub.Close(import_handle)

{
  "identifier": {
    "url": "file:///workspaces/duckdb/asam_ods_exd_api_duckdb/data/test.duckdb",
    "parameters": "mode=normalized"
  },
  "name": "test.duckdb",
  "groups": [
    {
      "name": "mea",
      "totalNumberOfChannels": "30000",
      "numberOfRows": "100",
      "channels": [
        {
          "name": "0",
          "dataType": "DT_DOUBLE"
        },
        {
          "id": "1",
          "name": "1",
          "dataType": "DT_DOUBLE"
        },
        {
          "id": "2",
          "name": "10",
          "dataType": "DT_DOUBLE"
        },
        {
          "id": "3",
          "name": "100",
          "dataType": "DT_DOUBLE"
        },
        {
          "id": "4",
          "name": "1000",
          "dataType": "DT_DOUBLE"
        },
        {
          "id": "5",
          "name": "10000",
          "dataType": "DT_DOUBLE"
        },
        {
          "id": "6",
          "name": "10001",
          "dataType": "DT_DOUBLE"
        },
        {
       

## Access Bulk Data

With the stored information the ASAM ODS server can access the bulk data from the EXD-API plugin

In [18]:
with grpc.insecure_channel(exd_api_plugin_url) as channel:
    stub = ods_external_data_pb2_grpc.ExternalDataReaderStub(channel)

    # info from physical storage
    access_group_id = 0
    access_channel_ids = [0, 1, 2]
    access_identifier = ods_external_data_pb2.Identifier(url=access_file_url, parameters=access_file_parameters)

    # open bulk access
    access_handle = stub.Open(access_identifier)
    try:
        request = ods_external_data_pb2.ValuesRequest(
            handle=access_handle, group_id=access_group_id, channel_ids=access_channel_ids
        )

        # read first chunk
        request.start = 0
        request.limit = 3
        values = stub.GetValues(request)
        print(MessageToJson(values))

        # read second chunk
        request.start = 3
        request.limit = 10
        values = stub.GetValues(request)
        print(MessageToJson(values))

    finally:
        stub.Close(access_handle)

_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:50051: Failed to connect to remote host: Connection refused"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_message:"failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:50051: Failed to connect to remote host: Connection refused", grpc_status:14}"
>